# World Stress Map 2025 — Data Validation

**Coordinate and azimuth quality assessment**

This notebook checks coordinate completeness and ranges, distinguishes numeric SHmax azimuths from special database codes, identifies potentially inconsistent records, and prepares a quality-filtered subset for subsequent mapping.

**Source:** [World Stress Map](https://www.world-stress-map.org/). The source CSV (`WSM_Database_2025.csv`) is obtained separately and accessed through a relative project path.

**Scope:** Automated checks identify *candidates for review*; they do not prove that a geographically plausible location or a recorded stress value is scientifically correct. The input CSV is read without modification.

## 1. Load the original data

In [1]:
import pandas as pd

df = pd.read_csv("WSM_Database_2025.csv", low_memory=False)
print(f"Observations: {len(df):,}; original columns: {df.shape[1]}")
df[["ID", "LAT", "LON", "AZI", "QUALITY", "REGIME"]].head()

Observations: 100,842; original columns: 40


,ID,LAT,LON,AZI,QUALITY,REGIME
0,wsm00015,36.880,30.670,0,C,U
1,wsm00016,40.180,29.100,0,B,NF
2,wsm00017,41.800,33.690,0,C,NF
3,wsm00025,56.821,12.714,17,B,TF
4,wsm00026,57.428,16.685,140,B,TS


## 2. Geographic coordinate checks

For this workflow, valid latitude is from **−90° to +90°** and longitude uses the **−180° to +180°** convention. Missing coordinates and out-of-range coordinates are counted separately.

In [2]:
def valid_latitude(lat):
    """Classify one latitude as Valid, Missing, or Invalid."""
    if pd.isna(lat):
        return "Missing"
    elif -90 <= lat <= 90:
        return "Valid"
    else:
        return "Invalid"


def valid_longitude(lon):
    """Classify one longitude using the -180 to 180 convention."""
    if pd.isna(lon):
        return "Missing"
    elif -180 <= lon <= 180:
        return "Valid"
    else:
        return "Invalid"

In [3]:
df["LAT_STATUS"] = df["LAT"].apply(valid_latitude)
df["LON_STATUS"] = df["LON"].apply(valid_longitude)

lat_counts = df["LAT_STATUS"].value_counts()
lon_counts = df["LON_STATUS"].value_counts()

coordinate_report = pd.DataFrame({
    "Status": ["Valid", "Invalid", "Missing"],
    "Latitude": [lat_counts.get(s, 0) for s in ["Valid", "Invalid", "Missing"]],
    "Longitude": [lon_counts.get(s, 0) for s in ["Valid", "Invalid", "Missing"]],
})
coordinate_report

,Status,Latitude,Longitude
0,Valid,100672,100672
1,Invalid,0,0
2,Missing,170,170


In [4]:
# Records with both coordinates present and inside the chosen ranges.
df_valid_coordinates = df[
    (df["LAT_STATUS"] == "Valid") &
    (df["LON_STATUS"] == "Valid")
].copy()

print(f"Valid coordinate pairs: {len(df_valid_coordinates):,}")
print(f"Excluded from coordinate-based mapping: {len(df) - len(df_valid_coordinates):,}")

Valid coordinate pairs: 100,672
Excluded from coordinate-based mapping: 170


## 3. SHmax azimuth availability

The `AZI` field contains values from **0° through 180°** for available axes in the inspected dataset, as well as the special value **999** associated with unavailable orientation. We classify explicit `NaN` separately from that special code. Values outside these cases are marked `Other` for review.

These categories describe field availability, **not** the scientific reliability of every observation.

In [5]:
def validate_azimuth(azi):
    """Classify a WSM azimuth field without changing the original value."""
    if pd.isna(azi):
        return "Missing"
    elif 0 <= azi <= 180:
        return "Available"
    elif azi == 999:
        return "Unavailable"
    else:
        return "Other"

In [6]:
df["AZI_STATUS"] = df["AZI"].apply(validate_azimuth)
azi_counts = df["AZI_STATUS"].value_counts()

azimuth_report = pd.DataFrame({
    "Status": ["Available", "Unavailable", "Missing", "Other"],
    "Azimuth": [azi_counts.get(s, 0) for s in ["Available", "Unavailable", "Missing", "Other"]],
})
azimuth_report

,Status,Azimuth
0,Available,89747
1,Unavailable,11095
2,Missing,0
3,Other,0


### Investigate a possible field inconsistency

During exploration, one record had `QUALITY = B` while its azimuth was `999`. That combination is **flagged for review**, not automatically corrected. It could have an explanation in the source record or publication.

In [7]:
flagged_quality_b = df[
    (df["AZI"] == 999) & (df["QUALITY"] == "B")
].copy()

flagged_quality_b[["ID", "LAT", "LON", "AZI", "QUALITY", "REGIME", "TYPE", "REF1", "COMMENT"]]

,ID,LAT,LON,AZI,QUALITY,REGIME,TYPE,REF1,COMMENT
66505,wsm250689,40.4,0.69,999,B,U,FMF,OLAIXX2025,NaN


## 4. Validated subsets for geospatial analysis

Retain observations with valid geographic coordinates and available azimuths, then select quality classes **A–C** for regional SHmax mapping. These filters generate derived subsets without modifying the original database.

In [8]:
df_mapping = df[
    (df["LAT_STATUS"] == "Valid") &
    (df["LON_STATUS"] == "Valid") &
    (df["AZI_STATUS"] == "Available")
].copy()

df_mapping_abc = df_mapping[
    df_mapping["QUALITY"].isin(["A", "B", "C"])
].copy()

mapping_report = pd.DataFrame({
    "Stage": [
        "All original records",
        "Valid coordinate pairs",
        "Available azimuths (all locations)",
        "Valid coordinates + available azimuth",
        "Valid coordinates + available azimuth + quality A–C",
    ],
    "Observations": [
        len(df),
        len(df_valid_coordinates),
        (df["AZI_STATUS"] == "Available").sum(),
        len(df_mapping),
        len(df_mapping_abc),
    ],
})
mapping_report

,Stage,Observations
0,All original records,100842
1,Valid coordinate pairs,100672
2,Available azimuths (all locations),89747
3,Valid coordinates + available azimuth,89624
4,Valid coordinates + available azimuth + qualit...,77364


In [9]:
df_mapping_abc["QUALITY"].value_counts().reindex(["A", "B", "C"], fill_value=0)

QUALITY
A     1757
B     1897
C    73710
Name: count, dtype: int64

## 5. Basic tests

These checks include boundary values, missing values, and deliberately invalid inputs. They test the functions even if the current source dataset contains no out-of-range geographic coordinates.

In [10]:
assert valid_latitude(-90) == "Valid"
assert valid_latitude(90) == "Valid"
assert valid_latitude(91) == "Invalid"
assert valid_latitude(float("nan")) == "Missing"

assert valid_longitude(-180) == "Valid"
assert valid_longitude(180) == "Valid"
assert valid_longitude(-181) == "Invalid"
assert valid_longitude(float("nan")) == "Missing"

assert validate_azimuth(45) == "Available"
assert validate_azimuth(999) == "Unavailable"
assert validate_azimuth(220) == "Other"
assert validate_azimuth(float("nan")) == "Missing"

print("All 12 function tests passed.")

All 12 function tests passed.


In [11]:
# Basic integrity checks: count totals and filtering conditions.
assert coordinate_report["Latitude"].sum() == len(df)
assert coordinate_report["Longitude"].sum() == len(df)
assert azimuth_report["Azimuth"].sum() == len(df)
assert df_mapping_abc["QUALITY"].isin(["A", "B", "C"]).all()
assert df_mapping_abc["AZI_STATUS"].eq("Available").all()

print("Report and subset checks passed.")

Report and subset checks passed.


## 6. Findings and limitations

For the WSM 2025 export (**100,842 records, 40 original columns**), the analysis found:

- **100,672** observations with valid coordinate pairs under our selected coordinate conventions; **170** records without coordinates; **0** out-of-range coordinates.
- **89,747** observations with azimuth in the recorded **0°–180°** range; **11,095** with special value **999**; no `NaN` or other numeric azimuth values.
- **89,624** observations with both valid coordinates and an available azimuth.
- **77,364** mapping candidates after restricting to quality **A–C** (**A: 1,757; B: 1,897; C: 73,710**).
- One potential inconsistency (record **`wsm250689`**, quality **B** with `AZI = 999`) identified for further investigation, **without modifying it**.

These results are specific to this dataset version; the preceding code recalculates them when the input changes. Range validity does not guarantee positional accuracy. Special codes and quality categories should be interpreted according to the official WSM documentation. All subsets are derived in memory; the original CSV is preserved.